In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import math

In [40]:
x = torch.rand((4,6))
linear = nn.Linear(6, 10)
x

tensor([[0.9050, 0.7355, 0.8625, 0.2043, 0.9714, 0.2714],
        [0.3971, 0.9656, 0.2945, 0.6695, 0.6346, 0.5553],
        [0.3334, 0.7089, 0.2274, 0.2210, 0.2527, 0.5952],
        [0.4662, 0.1684, 0.9960, 0.9107, 0.5031, 0.0338]])

In [41]:
# GLOBALS 
token_amount = 50
B,T,C  = 4, 8 ,32 # Batch size, time steps (seq length), channels (embedding vector size per token)
seq_len = 8


In [42]:
x = linear(x)
x

tensor([[-0.2615, -0.1284,  0.9042,  0.2749,  0.2675,  1.3172,  0.0358,  0.9997,
          0.6054, -0.5180],
        [-0.2479, -0.4178,  0.7841,  0.4813,  0.1193,  0.8640, -0.0221,  0.9312,
          0.3013, -0.4768],
        [-0.4333, -0.3573,  0.5456,  0.1461,  0.1180,  0.6191,  0.1089,  0.7990,
          0.0189, -0.3545],
        [ 0.1133, -0.5920,  0.8450,  0.1020,  0.2330,  1.0725,  0.2189,  0.4053,
          0.2734, -0.5004]], grad_fn=<AddmmBackward0>)

In [43]:
ten = 2 * torch.rand((5,5)) - 1
ten

tensor([[ 0.8741,  0.9376, -0.2111, -0.9331,  0.8980],
        [-0.9525, -0.9618,  0.9389,  0.8443,  0.4676],
        [ 0.7187,  0.0038,  0.8482,  0.8110, -0.9459],
        [ 0.3314, -0.4500,  0.8900,  0.7615, -0.9516],
        [-0.7607,  0.1240, -0.9404, -0.9558,  0.2267]])

In [44]:
approx = 'None' # 'tanh'
gelu = nn.GELU()
y = gelu(ten)
y

tensor([[ 0.7071,  0.7743, -0.0879, -0.1637,  0.7322],
        [-0.1623, -0.1617,  0.7756,  0.6760,  0.3179],
        [ 0.5490,  0.0019,  0.6802,  0.6418, -0.1628],
        [ 0.2087, -0.1469,  0.7238,  0.5916, -0.1624],
        [-0.1700,  0.0681, -0.1632, -0.1621,  0.1337]])

In [45]:
relu = nn.ReLU()
y_rel = relu(ten)
y_rel

tensor([[0.8741, 0.9376, 0.0000, 0.0000, 0.8980],
        [0.0000, 0.0000, 0.9389, 0.8443, 0.4676],
        [0.7187, 0.0038, 0.8482, 0.8110, 0.0000],
        [0.3314, 0.0000, 0.8900, 0.7615, 0.0000],
        [0.0000, 0.1240, 0.0000, 0.0000, 0.2267]])

In [46]:
torch.rand((3,6))

tensor([[0.4370, 0.9307, 0.2230, 0.1457, 0.3407, 0.7749],
        [0.4032, 0.6588, 0.2196, 0.9822, 0.6906, 0.4146],
        [0.6481, 0.8402, 0.9362, 0.5510, 0.6365, 0.7838]])

In [47]:
B,T,C  = 4, 8 ,32
x = torch.rand(B,T,C)

In [48]:
x.shape

torch.Size([4, 8, 32])

In [49]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

In [50]:
k = key(x)
q = query(x)
k.shape, q.shape

(torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [51]:
wei = q @ k.transpose(-1, -2)
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [52]:
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[[ 0.6245,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.6527,  0.5303,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.3744,  0.1379,  0.4509,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.5776,  0.8419,  0.8981,  0.7711,    -inf,    -inf,    -inf,
             -inf],
         [ 0.3627,  0.6352,  0.4698,  0.3876,  0.3851,    -inf,    -inf,
             -inf],
         [ 0.5100,  0.6926,  0.7761,  0.3091,  0.4188,  0.4211,    -inf,
             -inf],
         [ 0.2970,  0.4500,  0.4580,  0.0548,  0.2902,  0.2879,  0.4572,
             -inf],
         [ 0.1722,  0.3920,  0.3025,  0.2305,  0.1103,  0.3692,  0.3769,
           0.2073]],

        [[ 1.0320,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 0.7545,  0.3724,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf],
         [ 1.0808,  0.9630,  0.8220,    -inf,    -inf,    -inf,    -

In [53]:
wei = F.softmax(wei, dim=1)
wei

tensor([[[0.1475, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1517, 0.1404, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1149, 0.0948, 0.1463, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1407, 0.1918, 0.2288, 0.2956, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1135, 0.1559, 0.1491, 0.2015, 0.2700, 0.0000, 0.0000, 0.0000],
         [0.1315, 0.1652, 0.2025, 0.1863, 0.2793, 0.3540, 0.0000, 0.0000],
         [0.1063, 0.1296, 0.1473, 0.1444, 0.2456, 0.3099, 0.5201, 0.0000],
         [0.0938, 0.1223, 0.1261, 0.1722, 0.2051, 0.3361, 0.4799, 1.0000]],

        [[0.1464, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1109, 0.1085, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1537, 0.1959, 0.1845, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1213, 0.1435, 0.1595, 0.2018, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0888, 0.1186, 0.1426, 0.1597, 0.2250, 0.0000, 0.0000, 0.0000],
         [0.0882, 0.102

In [54]:
wei.shape

torch.Size([4, 8, 8])

In [55]:
v = value(x)

In [56]:
v.shape

torch.Size([4, 8, 16])

In [57]:
att = wei @ v

In [58]:
att.shape

torch.Size([4, 8, 16])

In [59]:
test = torch.ones(5)

In [60]:
test[0:2] = test[:2] + 3

In [61]:
test.std()

tensor(1.6432)

In [62]:
x.shape

torch.Size([4, 8, 32])

In [63]:
emb = nn.Embedding(token_amount, C)

In [64]:
torch.manual_seed(47)
input_sequences = torch.randint(0, token_amount, (B, T))
input_sequences

tensor([[ 5, 48, 49,  4,  0, 21, 36,  1],
        [23,  2, 41, 25,  3, 49, 17,  8],
        [22, 40, 19, 11, 27, 21, 38, 42],
        [45, 28, 49, 49, 47, 20, 35, 40]])

In [65]:
# Here we're doing the indexing. For each sequence example, fetch the corresponding embedding matrix.
# `input_sequences` here should be B examples of T token sequences in which each T element is a token ID not some rand value
input_embedding = emb(input_sequences)
input_embedding.shape # B T C

torch.Size([4, 8, 32])

In [66]:
pos_embedding = nn.Embedding(seq_len, C) # For each position, we have an embedding vector.
pos = torch.arange(0, T)
positional_embeddings = pos_embedding(pos)
positional_embeddings.shape

torch.Size([8, 32])

In [67]:
positional_embeddings[0]

tensor([-0.5540, -0.1677, -1.7382, -0.9734, -0.2886, -0.9845, -0.1295, -0.8175,
         1.0130,  0.3874, -0.4618,  1.3403,  0.8922,  0.2661,  0.2014,  0.6140,
         0.1047, -0.1160, -0.5885,  0.0255, -1.2083,  0.2856,  0.7352, -0.6235,
         0.9029,  0.9697,  0.1173, -0.0266,  0.4993, -0.7550,  0.7418,  1.5704],
       grad_fn=<SelectBackward0>)

In [68]:
input_embedding[0][0]

tensor([ 0.2465, -1.4059,  0.4682,  1.0302,  1.0594,  1.0681, -0.4952, -3.1751,
         1.3070, -0.1600,  0.6188, -0.6398,  1.3498,  0.0549,  1.5595,  2.8717,
         1.0581,  0.7228, -0.5053, -0.6995,  1.9409, -0.1757, -0.5294, -0.6210,
        -0.2054,  1.5362,  0.9942, -1.1291,  0.8296, -0.2177,  0.2324, -3.1981],
       grad_fn=<SelectBackward0>)

In [69]:
x = input_embedding +  positional_embeddings
f'{x[0][0][0].item():4f}, {(1.3891 + -1.7554):4f}' # == (1.3891 + -1.7554)

'-0.307491, -0.366300'

In [70]:
test_tensor = torch.arange(0, 20).view(5, 4)
test_tensor

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15],
        [16, 17, 18, 19]])

In [71]:
test_tensor.split(2, 1)

(tensor([[ 0,  1],
         [ 4,  5],
         [ 8,  9],
         [12, 13],
         [16, 17]]),
 tensor([[ 2,  3],
         [ 6,  7],
         [10, 11],
         [14, 15],
         [18, 19]]))

In [72]:
tril = torch.tril(torch.ones(T,T))
mask = tril.masked_fill(tril == 0, float('-inf'))
mask

tensor([[1., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., -inf, -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., -inf, -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., -inf, -inf, -inf, -inf],
        [1., 1., 1., 1., 1., -inf, -inf, -inf],
        [1., 1., 1., 1., 1., 1., -inf, -inf],
        [1., 1., 1., 1., 1., 1., 1., -inf],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [73]:
@dataclass
class GPTConfig:
    seq_len: int = 8
    embedding_len = 32
    n_heads = 8
    n_blocks = 0

class SelfAttention(nn.Module):
    
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config
        self.n_heads = config.n_heads
        self.embedding_len = config.embedding_len
        # Each atten head processes an equal portion of the embedding so the numbers must be compatible.
        assert config.embedding_len % config.n_heads == 0
        self.qkv_lin = nn.Linear(config.embedding_len, config.embedding_len * 3)
        self.proj_lin = nn.Linear(config.embedding_len, config.embedding_len)

        # Add stencil here? better with register buffer so it's no grad

    def forward(self, x):
        # rough steps: get QKV > matmul q with k > get mask > apply mask > softmax 
        # > matmul softmax result (wei) with V -> project? (or is that done before?) 
        B, T, C = x.size() 
        # XXX: Would C being smaller than embed_len cause compatibility issue with n_heads?
        # NO!!! C disappears after linear matmul. But wouldn't it prevent matmul cus of mismatch? (B,C) @ (n_emb, n_emb) if C != n_emb > crash
        assert C <= self.config.embedding_len, "wtf?" 
        qkv = self.qkv_lin(x) # B T EMBED_LEN*3
        q, k, v = qkv.split(self.config.embedding_len, -1) # split over emb dim
        
        # Pre process for multihead attn. 
        # For each example, for each head, for each timestep we have headsize result
        q = q.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2) # (B, n_heads, T, head_size) 
        k = k.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        v = v.view(B, T, self.n_heads, self.n_heads // self.embedding_len).transpose(1,2)
        # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) = (B, n_heads, T, T)?
        wei = q @ k.transpose(-1, -2)
        with torch.no_grad():
            tril = torch.tril(torch.ones(T,T))
            wei = wei.masked_fill(tril == 0, float('-inf'))
            print(wei)

In [74]:
s_attn = SelfAttention(GPTConfig())

In [75]:
x = torch.rand(4, GPTConfig.seq_len, GPTConfig.embedding_len)
x.shape

torch.Size([4, 8, 32])

In [76]:
out = s_attn(x)

RuntimeError: shape '[4, 8, 8, 0]' is invalid for input of size 1024

In [ ]:
math.exp(1)

2.718281828459045

In [ ]:
test_softmax = torch.arange(0, 6).float().view(2,3)
test_softmax

tensor([[0., 1., 2.],
        [3., 4., 5.]])

In [ ]:
F.softmax(test_softmax, dim=0)

tensor([[0.0474, 0.0474, 0.0474],
        [0.9526, 0.9526, 0.9526]])

In [ ]:
tst_view = torch.rand(4,8,8,4)
tst_view.shape

torch.Size([4, 8, 8, 4])

In [ ]:
tst_view.view(4,8,32)

tensor([[[0.8335, 0.3510, 0.7374,  ..., 0.4986, 0.9729, 0.9660],
         [0.8035, 0.8097, 0.3513,  ..., 0.1751, 0.0339, 0.7793],
         [0.0822, 0.3366, 0.8414,  ..., 0.2126, 0.0264, 0.2305],
         ...,
         [0.3901, 0.8549, 0.7311,  ..., 0.5139, 0.6362, 0.9789],
         [0.7353, 0.4125, 0.5198,  ..., 0.0455, 0.6813, 0.9366],
         [0.3041, 0.1002, 0.4681,  ..., 0.0343, 0.2428, 0.8075]],

        [[0.5695, 0.8297, 0.3437,  ..., 0.2393, 0.3704, 0.6779],
         [0.3767, 0.1497, 0.0098,  ..., 0.7605, 0.2431, 0.5804],
         [0.8856, 0.1932, 0.2340,  ..., 0.4430, 0.2799, 0.4387],
         ...,
         [0.6070, 0.9912, 0.8454,  ..., 0.1552, 0.4035, 0.8872],
         [0.5107, 0.5700, 0.2241,  ..., 0.6545, 0.7925, 0.1229],
         [0.5824, 0.3939, 0.4118,  ..., 0.3477, 0.3773, 0.2346]],

        [[0.5576, 0.6844, 0.5592,  ..., 0.7349, 0.4985, 0.6342],
         [0.5589, 0.2784, 0.9481,  ..., 0.1934, 0.6645, 0.8061],
         [0.1573, 0.4511, 0.9218,  ..., 0.7916, 0.2409, 0.

In [ ]:
a = torch.randint(1,5, (2,2))
b = torch.randint(1,5, (2,))
a, b

(tensor([[1, 3],
         [4, 3]]),
 tensor([2, 1]))

In [ ]:
a = torch.randint(1,11, (3,1,3))
a

tensor([[[ 9,  1,  4]],

        [[10,  4,  6]],

        [[ 5, 10,  5]]])

In [ ]:
# torch.topk(a, 1, dim=0)
vals_tensor, indices_tensor = a.topk(1, dim=-1)
vals_tensor, indices_tensor

(tensor([[[ 9]],
 
         [[10]],
 
         [[10]]]),
 tensor([[[0]],
 
         [[0]],
 
         [[1]]]))

In [ ]:
sampling = torch.rand(4,4,5)
sampling

tensor([[[0.8353, 0.6266, 0.2309, 0.8413, 0.9017],
         [0.0997, 0.0843, 0.7229, 0.4912, 0.5008],
         [0.0086, 0.1117, 0.9843, 0.5537, 0.0097],
         [0.2016, 0.3952, 0.8641, 0.9786, 0.9982]],

        [[0.5678, 0.8303, 0.5466, 0.1510, 0.3714],
         [0.4989, 0.6035, 0.7160, 0.1658, 0.6305],
         [0.9044, 0.8954, 0.3396, 0.7829, 0.6020],
         [0.6608, 0.5096, 0.8899, 0.1432, 0.3919]],

        [[0.7101, 0.5054, 0.0013, 0.2828, 0.0726],
         [0.9530, 0.9228, 0.4556, 0.0929, 0.6731],
         [0.9948, 0.7785, 0.6316, 0.5736, 0.3361],
         [0.3766, 0.2125, 0.2158, 0.7904, 0.1163]],

        [[0.1224, 0.9106, 0.5472, 0.4197, 0.6174],
         [0.5278, 0.7779, 0.2010, 0.1283, 0.7544],
         [0.8307, 0.5393, 0.0736, 0.5909, 0.7626],
         [0.0420, 0.4902, 0.3697, 0.8389, 0.4133]]])

In [ ]:
top_vals, top_idx = sampling.topk(2)
top_vals.shape


torch.Size([4, 4, 2])

In [ ]:
top_vals = top_vals.view(16,2)
top_vals


tensor([[0.9017, 0.8413],
        [0.7229, 0.5008],
        [0.9843, 0.5537],
        [0.9982, 0.9786],
        [0.8303, 0.5678],
        [0.7160, 0.6305],
        [0.9044, 0.8954],
        [0.8899, 0.6608],
        [0.7101, 0.5054],
        [0.9530, 0.9228],
        [0.9948, 0.7785],
        [0.7904, 0.3766],
        [0.9106, 0.6174],
        [0.7779, 0.7544],
        [0.8307, 0.7626],
        [0.8389, 0.4902]])

In [ ]:
sample_idx = top_vals.multinomial(1)
sample_idx

tensor([[0],
        [1],
        [1],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [1],
        [0],
        [1],
        [0],
        [0],
        [1]])

In [ ]:
top_idx.shape, top_idx

(torch.Size([4, 4, 2]),
 tensor([[[4, 3],
          [2, 4],
          [2, 3],
          [4, 3]],
 
         [[1, 0],
          [2, 4],
          [0, 1],
          [2, 0]],
 
         [[0, 1],
          [0, 1],
          [0, 1],
          [3, 0]],
 
         [[1, 4],
          [1, 4],
          [0, 4],
          [3, 1]]]))

In [ ]:
x = torch.tensor([[1,2],[3,4]])
x

tensor([[1, 2],
        [3, 4]])

In [ ]:
x.gather(0, torch.tensor([[1],[0]]))

tensor([[3],
        [1]])

In [ ]:
proba_dist = torch.rand(3,3,3)

proba_dist

tensor([[[0.8229, 0.2031, 0.1697],
         [0.3440, 0.7221, 0.5861],
         [0.2359, 0.1123, 0.0641]],

        [[0.1017, 0.2499, 0.8767],
         [0.3932, 0.5220, 0.7646],
         [0.2997, 0.4182, 0.9184]],

        [[0.4845, 0.6471, 0.8710],
         [0.0971, 0.0483, 0.8541],
         [0.1856, 0.2468, 0.9583]]])

In [ ]:
proba_dist

tensor([[[0.8229, 0.0000, 0.0000],
         [0.3440, 0.7221, 0.5861],
         [0.2359, 0.1123, 0.0641]],

        [[0.1017, 0.2499, 0.8767],
         [0.3932, 0.5220, 0.7646],
         [0.2997, 0.4182, 0.9184]],

        [[0.4845, 0.6471, 0.8710],
         [0.0971, 0.0483, 0.8541],
         [0.1856, 0.2468, 0.9583]]])

In [ ]:
proba_dist.view(9,3).multinomial(1)

tensor([[0],
        [2],
        [0],
        [2],
        [0],
        [1],
        [0],
        [2],
        [2]])

In [ ]:
probas = torch.rand(4,8,8)
probas

tensor([[[0.8170, 0.4310, 0.9232, 0.5910, 0.4819, 0.7750, 0.0494, 0.1389],
         [0.7961, 0.7262, 0.2960, 0.6552, 0.1324, 0.4364, 0.9153, 0.1001],
         [0.9804, 0.4660, 0.8022, 0.8772, 0.9769, 0.5359, 0.5536, 0.2415],
         [0.0773, 0.4625, 0.8495, 0.5343, 0.1302, 0.3831, 0.8131, 0.8265],
         [0.1747, 0.9002, 0.3008, 0.0250, 0.5452, 0.7241, 0.1347, 0.6980],
         [0.9654, 0.2491, 0.8830, 0.3255, 0.6728, 0.8577, 0.8540, 0.8510],
         [0.2278, 0.5066, 0.5974, 0.0041, 0.7831, 0.4098, 0.4182, 0.5655],
         [0.0913, 0.6629, 0.9711, 0.1880, 0.5733, 0.9914, 0.5295, 0.6218]],

        [[0.8816, 0.1975, 0.5204, 0.3232, 0.7370, 0.0100, 0.2000, 0.3444],
         [0.2017, 0.4089, 0.1429, 0.4213, 0.2477, 0.4108, 0.0491, 0.1749],
         [0.3575, 0.0399, 0.1748, 0.1632, 0.0771, 0.9797, 0.9856, 0.9972],
         [0.9726, 0.4207, 0.9927, 0.2384, 0.4596, 0.1124, 0.6790, 0.7810],
         [0.2835, 0.7390, 0.5460, 0.3075, 0.9895, 0.3080, 0.8106, 0.3167],
         [0.4916, 0.564

In [ ]:
top_v, top_i = probas.topk(2, dim = -1)
top_v.shape

torch.Size([4, 8, 2])

In [ ]:
top_i.view(4*8, 2)[:2]

tensor([[2, 0],
        [6, 0]])

In [ ]:
picks = top_v.view(4*8, 2).multinomial(1)
picks.shape
picks = picks.view(4, 8, 1)
picks

tensor([[[0],
         [1],
         [0],
         [0],
         [0],
         [0],
         [1],
         [0]],

        [[0],
         [1],
         [0],
         [1],
         [0],
         [1],
         [0],
         [0]],

        [[1],
         [1],
         [1],
         [0],
         [0],
         [0],
         [1],
         [0]],

        [[0],
         [1],
         [1],
         [1],
         [1],
         [1],
         [1],
         [0]]])

In [ ]:
top_i.shape, top_i[:1]

(torch.Size([4, 8, 2]),
 tensor([[[2, 0],
          [6, 0],
          [0, 4],
          [2, 7],
          [1, 5],
          [0, 2],
          [4, 2],
          [5, 2]]]))

In [ ]:
top_i.gather(-1, picks)

tensor([[[2],
         [0],
         [0],
         [2],
         [1],
         [0],
         [2],
         [5]],

        [[0],
         [5],
         [7],
         [0],
         [4],
         [1],
         [1],
         [5]],

        [[3],
         [6],
         [7],
         [7],
         [7],
         [3],
         [0],
         [2]],

        [[0],
         [4],
         [2],
         [6],
         [4],
         [3],
         [6],
         [1]]])

In [93]:
w = torch.tensor([2.0], requires_grad=True)
x = torch.tensor([3.0])
y = w * x 
y

tensor([6.], grad_fn=<MulBackward0>)

In [82]:
y.requires_grad

True

In [86]:
vals, idx = torch.topk(y, 1)
vals.requires_grad, idx.requires_grad

(True, False)

In [88]:
i = torch.multinomial(torch.softmax(vals, dim=0), 1)
i.requires_grad

False

In [91]:
loss = vals[i].sum()
loss.backward()

In [96]:
print(w.grad)

None


In [108]:
# w = torch.tensor([2,3,5], dtype=float)
w = torch.rand(2,2)
x = torch.rand(3,2)

In [145]:
y = x @ w
y.requires_grad

False

In [ ]:
# Pytorch flips timestep and channel dims.
# BTC for logits (input of the CE) becomes BCT.
logits = torch.rand(4,10,5)
y = torch.randint(10, (4,5))
loss = F.cross_entropy(logits, y)

In [469]:
torch.tensor(5).shape

torch.Size([])

In [480]:
import numpy as np
lst = list(range(10))
arr = np.arange(10, dtype=np.uint16)
np2torch = torch.from_numpy(arr)
tens = torch.tensor(lst, dtype=torch.uint16)
tens = tens.to('cuda') if torch.cuda.is_available() else tens
# lst